# Stage 6 - Retention Planner (similarity-driven, lift-based)

This stage turns a churn prediction into a retention plan, and cosine similarity
helps **choose** the actions - not just back them up.

Workflow:

```text
Model predicts churn
  -> SHAP gives the risk factors
  -> cosine similarity finds comparable customers who STAYED and who CHURNED
  -> for each actionable feature, compare how common the better value is
     among retained vs churned neighbors (a lift)
  -> generate and prioritise retention actions
```

A feature becomes a recommendation when the model flags it as raising risk **and**
the better value is meaningfully more common among similar retained customers than
among similar churned customers - a minimum **lift** (>= 10 percentage points) and
**support** (held by >= 30% of retained neighbors), not just a 60% majority.

All logic lives in `src/retention/`; this notebook calls it.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
project_root = here if (here / "src").exists() else here.parent
sys.path.insert(0, str(project_root))

import json
import warnings

import pandas as pd

from src.config import (
    DATA_PROCESSED_DIR, MODELS_DIR, MODEL_PATH, DECISION_THRESHOLD_PATH,
    SIMILARITY_MIN_THRESHOLD, SIMILARITY_MAX_NEIGHBORS,
)
from src.data.split import split_data
from src.features.preprocessor import load_preprocessor
from src.models.train import load_model
from src.explainability.explainer import build_explainer, explain_customer
from src.similarity.vectorizer import build_customer_vectors, vectorize_one
from src.similarity.search import find_similar_retained, find_similar_churned
from src.retention.planner import build_retention_plan

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

customers = pd.read_csv(DATA_PROCESSED_DIR / "telco_clean.csv")
preprocessor = load_preprocessor(MODELS_DIR / "preprocessor.joblib")
model = load_model(MODEL_PATH)
decision_threshold = json.loads(DECISION_THRESHOLD_PATH.read_text())["threshold"]

explainer = build_explainer(model, split_data(customers).X_train, decision_threshold)
customer_vectors = build_customer_vectors(preprocessor, customers)


def make_plan(new_customer):
    """Run the full lift-based pipeline for one customer."""
    new_vector = vectorize_one(preprocessor, new_customer)
    retained = find_similar_retained(
        customer_vectors, new_vector,
        top_k=SIMILARITY_MAX_NEIGHBORS, min_similarity=SIMILARITY_MIN_THRESHOLD,
    )
    churned = find_similar_churned(
        customer_vectors, new_vector,
        top_k=SIMILARITY_MAX_NEIGHBORS, min_similarity=SIMILARITY_MIN_THRESHOLD,
    )
    explanation = explain_customer(explainer, new_customer)
    return build_retention_plan(explanation, new_customer, retained, churned, customer_vectors)


def show_plan(plan):
    """Print a retention plan in a readable way."""
    print(f"Churn probability : {plan['probability']:.1%}")
    print(f"Risk level        : {plan['risk_level']}")
    print(f"Evidence mode     : {plan['evidence_mode']} "
          f"({plan['retained_count']} retained, {plan['churned_count']} churned compared)")
    print()
    if not plan["recommendations"]:
        print("No evidence-backed retention actions for this customer.")
    for i, rec in enumerate(plan["recommendations"], start=1):
        print(f"{i}. [{rec['confidence']} confidence, score {rec['combined_score']:.2f}] {rec['action']}")
        print(f"   Current -> Suggested : {rec['current_value']} -> {rec['suggested_alternative']}")
        print(f"   Model evidence       : raises churn risk (contribution {rec['model_contribution']:+.2f})")
        if rec["mode"] == "lift":
            print(f"   Retained vs churned  : {rec['retained_rate']:.0%} vs {rec['churned_rate']:.0%} "
                  f"(lift +{rec['lift']:.0%}, support {rec['support']:.0%})")
        elif rec["mode"] == "majority":
            print(f"   Retained agreement   : {rec['retained_rate']:.0%} (too few churned peers for a lift)")
        else:
            print(f"   Similarity evidence  : none - model-only fallback")
        print()
    print("Note:", plan["disclaimer"])

## 1. A high-risk customer

In [2]:
high_risk_customer = pd.DataFrame(
    [
        {
            "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No",
            "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
            "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
            "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
            "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
            "PaymentMethod": "Electronic check", "MonthlyCharges": 95.0, "TotalCharges": 190.0,
        }
    ]
)

show_plan(make_plan(high_risk_customer))

Churn probability : 71.8%
Risk level        : High
Evidence mode     : lift (20 retained, 20 churned compared)

1. [High confidence, score 0.74] Provide onboarding help and an early loyalty benefit.
   Current -> Suggested : 2 -> onboarding and loyalty support (similar stayers' median tenure: 7 months)
   Model evidence       : raises churn risk (contribution +1.71)
   Retained vs churned  : 80% vs 30% (lift +49%, support 80%)

Note: Personalized recommendations based on model risk factors and patterns among similar retained customers. The dataset does not record which offers were applied, so these are not proof that an action caused any customer to stay.


**Conclusion:** The lift comparison drives the plan. For this customer the
strongest signal is **tenure**: similar retained customers have a much longer
tenure than similar churned customers (a large positive lift), so the action is
onboarding and loyalty support. A feature is only recommended when its better value
genuinely separates similar stayers from similar churners - so we avoid suggesting
changes that comparable churned customers also had.

## 2. A low-risk customer

In [3]:
low_risk_customer = pd.DataFrame(
    [
        {
            "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes",
            "tenure": 64, "PhoneService": "Yes", "MultipleLines": "Yes",
            "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
            "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "No",
            "StreamingMovies": "No", "Contract": "Two year", "PaperlessBilling": "No",
            "PaymentMethod": "Bank transfer (automatic)", "MonthlyCharges": 60.0, "TotalCharges": 3840.0,
        }
    ]
)

show_plan(make_plan(low_risk_customer))

Churn probability : 0.9%
Risk level        : Low
Evidence mode     : majority (20 retained, 3 churned compared)

No evidence-backed retention actions for this customer.
Note: Personalized recommendations based on model risk factors and patterns among similar retained customers. The dataset does not record which offers were applied, so these are not proof that an action caused any customer to stay.


**Conclusion:** A low-risk customer with few model-flagged risk factors gets no
actions. The planner stays quiet when there is nothing evidence-backed to do.

## 3. Why lift, and how the thresholds were chosen

The nearest retained customers are similar precisely because they **share** the
query's profile, so a plain "most similar stayers have a longer contract" rule
almost never fires (the similar stayers are often month-to-month too). The fix is
to compare retained neighbors against **churned** neighbors:

> Recommend a better value when it is more common among similar retained customers
> than among similar churned customers.

On a sample of 250 high-risk customers, sweeping the thresholds gave:

| min lift | min support | coverage | typical features |
|---|---|---|---|
| 0.05 | 0.20 | 76% | tenure, payment method, online backup, paperless... |
| **0.10** | **0.30** | **65%** | tenure, payment method, monthly charges, device protection... |
| 0.20 | 0.40 | 42% | tenure, payment method... |

We chose **lift >= 0.10 and support >= 0.30**: a real differentiator (the better
value is at least 10 points more common among similar stayers than churners) that
is actually present in the retained group. This is not a lowered majority rule - it
is a different, stronger test.

Each recommendation is prioritised by a blend of **model risk contribution, lift,
similarity quality, and retained support**, and shown with its retained rate,
churned rate, lift, support, and a High/Medium/Low confidence. If there are too few
similar churned customers to compute a lift, the planner falls back to agreement
among retained neighbors; with no retained neighbors at all, it falls back to
model-supported actions, clearly labelled.

## Honest framing

- Recommendations describe model risk factors and patterns among similar customers
  - **not proven causes**.
- The dataset does not record which retention offers were applied, so similar
  retained customers are comparable profiles, **not proof an offer worked**.
- Protected and personal features (gender, senior-citizen status, partner,
  dependents, customer ID) are never recommended.

## Summary of Stage 6

1. Cosine similarity now **chooses** actions via a retained-vs-churned **lift**: a
   feature is recommended only when its better value is meaningfully more common
   among similar stayers than similar churners (lift >= 10pp, support >= 30%).
2. Both neighbor groups are selected by full-vector cosine similarity (>= 0.80, up
   to 20 each) and weighted by similarity score.
3. Priority blends model contribution, lift, similarity quality, and support; each
   recommendation shows the retained rate, churned rate, lift, support, and
   confidence.
4. Fallbacks keep the old behavior: retained-agreement when churned peers are
   scarce, model-only when no retained peers qualify.
5. Only actionable features are recommended; protected features never are.

**The dashboard** renders this same plan on its Retention Actions page.